# 10 — Human in the Loop

**Mode:** Live  
**Level:** Advanced  
**Estimated time:** 40 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

Three real OpenAI-generated tool calls, each paused before execution. You will approve one, edit one, reject one, inspect durable SQLite state from a separate process, and resume every suspended run.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(10, 'Human in the Loop')


## Prerequisites and setup

**Course prerequisites:** Complete `course-05-tool-use`, `course-09-model-runtime`.

**Execution requirements:** Live OpenAI. Set `OPENAI_API_KEY` and `PRAVAL_OPENAI_MODEL`. The selected model must support tool calling. This notebook intentionally makes several bounded provider calls and runs only through protected manual certification.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Require approval through ToolSpec policy.
- Observe interruption before any side effect occurs.
- Persist and inspect suspended runs across a process boundary.
- Approve, edit, reject, and resume with structural assertions.


## Mental model

Human-in-the-loop is a durable state transition, not an `input()` prompt. A real model requests a tool. Praval stores the request and continuation, raises `InterventionRequired`, and executes nothing. A reviewer then approves the original arguments, edits them, or rejects the call. Resume continues from the stored state.


In [ ]:
show_flow(
('Model', 'requests tool', 'provider'),
('Praval', 'persist and pause', 'reef'),
('Human', 'approve, edit, reject', 'human'),
('Runtime', 'resume safely', 'agent'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Create durable HITL storage

All decisions share one temporary SQLite file. Configuration displays the model but never the API key.


In [ ]:
import json
import sqlite3
import subprocess
import sys
import tempfile

values = require_env("OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL")
workspace = tempfile.TemporaryDirectory(prefix="praval-hitl-notebook-")
database = str(Path(workspace.name) / "hitl.sqlite3")
show_json(
    {"provider": "openai", "model": values["PRAVAL_OPENAI_MODEL"],
     "database": "temporary SQLite"},
    "HITL configuration",
)


### What just happened?

The durable store has an isolated path that can be opened by another process.

### Why this matters

Approval state must survive longer than a notebook cell or web request.


### Define the guarded capability

Every Agent will receive the same strict, high-risk schema. The execution list is the proof that no tool runs before a decision.


In [ ]:
from praval import Agent, InterventionRequired, ToolSpec

executions = []
guarded_spec = ToolSpec(
    name="guarded_multiply",
    description="Multiply two integers; always use this for multiplication.",
    parameters={
        "type": "object",
        "properties": {
            "a": {"type": "integer"}, "b": {"type": "integer"},
        },
        "required": ["a", "b"], "additionalProperties": False,
    },
    strict=True, requires_approval=True, risk_level="high",
    approval_reason="The notebook operator must approve this execution.",
)


def make_agent(decision):
    agent = Agent(
        f"course-hitl-{decision}", provider="openai",
        model=values["PRAVAL_OPENAI_MODEL"], hitl_enabled=True,
        hitl_db_path=database,
        config={"temperature": 0, "max_output_tokens": 64, "timeout": 60},
    )
    def guarded_multiply(a: int, b: int) -> int:
        executions.append({"decision": decision, "a": a, "b": b,
                           "result": a * b})
        stage("tool", "executed", f"{decision}: {a} × {b}")
        return a * b
    agent.add_tool_spec(guarded_spec, guarded_multiply)
    return agent


### What just happened?

The ToolSpec carries both argument validation and approval policy. Each handler records its decision label if it actually executes.

### Why this matters

A model prompt alone cannot enforce approval; the runtime policy must gate the handler.


### Interrupt three real model-generated calls

Each Agent asks the configured OpenAI model to call the guarded tool. A run is valid only if Praval interrupts and execution remains empty.


In [ ]:
cases = {
    "approve": {"requested": (2, 3), "edited": None},
    "edit": {"requested": (4, 5), "edited": {"a": 7, "b": 8}},
    "reject": {"requested": (9, 9), "edited": None},
}
agents = {}
pending = {}
for decision, case in cases.items():
    agent = make_agent(decision)
    agents[decision] = agent
    a, b = case["requested"]
    try:
        agent.generate(
            f"You must call guarded_multiply with a={a} and b={b}. "
            "Do not calculate the answer yourself."
        )
    except InterventionRequired as interruption:
        requests = agent.get_pending_interventions(run_id=interruption.run_id)
        assert len(requests) == 1
        pending[decision] = {"exception": interruption, "request": requests[0]}
        stage("HITL", "interrupted", decision)
    else:
        raise AssertionError("The real model did not produce the required tool call")

assert executions == [] and set(pending) == set(cases)


### What just happened?

All three requests are durable and pending, and none of the guarded handlers has run.

### Why this matters

The interruption boundary is the safety property: the side effect must remain impossible until a recorded decision exists.


In [ ]:
show_json(
    {
        name: {
            "run_id": item["request"].run_id,
            "tool": item["request"].tool_name,
            "arguments": item["request"].original_args,
            "status": item["request"].status.value,
        }
        for name, item in pending.items()
    },
    "Persisted pending interventions",
)


### Inspect persistence from another process

A separate Python interpreter opens the SQLite file and counts pending interventions and suspended runs. It does not import the notebook process state.


In [ ]:
inspection_code = (
    'import json, sqlite3, sys; '
    'c=sqlite3.connect(sys.argv[1]); '
    'p=c.execute("select count(*) from interventions where status=\'PENDING\'").fetchone()[0]; '
    'r=c.execute("select count(*) from suspended_runs where status=\'pending\'").fetchone()[0]; '
    'print(json.dumps({"pending_interventions": p, "suspended_runs": r}))'
)
completed = subprocess.run(
    [sys.executable, "-c", inspection_code, database],
    check=True, capture_output=True, text=True, timeout=15,
)
persisted = json.loads(completed.stdout)
assert persisted == {"pending_interventions": 3, "suspended_runs": 3}
show_json(persisted, "State observed by a separate process")


### What just happened?

The second process saw all pending work using only the database path.

### Why this matters

Durability lets approval happen in another service or at a later time without keeping the original request thread alive.


### Apply approve, edit, and reject decisions

The approved call keeps model arguments, the edited call replaces them, and the rejected call records a reason. Decisions are persisted before resume.


In [ ]:
approved = pending["approve"]["request"]
agents["approve"].approve_intervention(
    approved.id, reviewer="notebook-operator"
)
edited = pending["edit"]["request"]
agents["edit"].approve_intervention(
    edited.id, reviewer="notebook-operator",
    edited_args=cases["edit"]["edited"],
)
rejected = pending["reject"]["request"]
agents["reject"].reject_intervention(
    rejected.id, reviewer="notebook-operator",
    reason="Demonstrating the bounded reject path",
)
stage("human", "decisions recorded", "approve, edit, reject")


### What just happened?

Each pending row now has a reviewer, decision, and timestamp; edited arguments or rejection reason are explicit.

### Why this matters

Auditability depends on recording who decided what, not only on the eventual tool result.


### Resume every run

Resume reconstructs the provider-neutral request and continuation, applies the decision, and lets the provider finish its response.


In [ ]:
resumed = {}
for decision, item in pending.items():
    resumed[decision] = agents[decision].resume_run(item["request"].run_id)
    stage("runtime", "resumed", decision)

approved_execution = next(item for item in executions if item["decision"] == "approve")
edited_execution = next(item for item in executions if item["decision"] == "edit")
assert approved_execution["result"] == 6
assert edited_execution == {
    "decision": "edit", "a": 7, "b": 8, "result": 56,
}
assert not any(item["decision"] == "reject" for item in executions)
assert all(value.strip() for value in resumed.values())
show_json({"executions": executions, "responses": resumed}, "Resumed outcomes")
show_timeline()


### What just happened?

Only approved and edited handlers ran. The edited handler received the reviewer arguments, and rejection produced no tool side effect.

### Why this matters

Resume must honor the stored decision exactly once and keep the terminal result inspectable.


## Your turn

Add a low-risk read-only tool that does not require approval and compare its execution path with `guarded_multiply`. Explain why risk and approval are related but not identical fields.


In [ ]:
# Define a ToolSpec(requires_approval=False, risk_level="low").
# Ask the real model to call it and confirm no InterventionRequired is raised.
# Keep the tool deterministic and read-only.


## Common mistake

**Mistake:** Executing the tool first and asking a human to confirm afterward.

**Correction:** Persist the model's requested call and stop before invoking the handler. Post-hoc confirmation is an audit log, not a safety gate.


<details>
<summary>Under the hood</summary>

ModelRuntime serializes the request, response, current tool index, accumulated results, and intervention ID. HITLStore uses transactional SQLite updates. On resume, the runtime reconstructs neutral objects, applies the decision, runs or rejects the tool, and continues the provider tool loop.

</details>


## Recap

- HITL pauses real model-generated tool calls before execution.
- Pending calls and continuation state are durable.
- Approve, edit, and reject have distinct execution semantics.
- Resume completes the stored run and leaves an auditable decision trail.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
for agent in agents.values():
    agent.close()
workspace.cleanup()
stage("agent", "closed", "HITL database removed")


### Next lesson

Continue to `11_mcp_tools.ipynb` to discover remote tool schemas and attach async-only handlers to this same Agent and HITL path.
